### Task 1.

Реализуйте функцию add_experiment.

**Шаблон решения**

In [ ]:
def add_experiment(experiment, buckets):
    """Проверяет можно ли добавить эксперимент, добавляет если можно.

    Распределять эксперименты нужно так, чтобы умещалось как можно больше экспериментов.

    :param experiment (dict): параметры эксперимента, который нужно запустить.
        Ключи словаря:
            - id - идентификатор эксперимента.
            - buckets_count - необходимое количество бакетов.
            - conflicts - список идентификаторов экспериментов, которые нельзя проводить
                одновременно на одних и тех же пользователях.
    :param buckets (list[list[int]]): список бакетов, в каждом бакете перечислены
            идентификаторы экспериментов, которые в нём проводятся.

    :return (success, buckets):
        success (boolean) - можно ли добавить эксперимент, True - можно, иначе - False
        buckets (list[list[int]]) - обновлённый список бакетов с добавленным экспериментом,
            если эксперимент добавить можно.
    """
        # YOUR_CODE_HERE

**Пример**

In [ ]:
total_buckets_count = 4
buckets = [[] for _ in range(total_buckets_count)]
success, buckets = add_experiment({'id': 0, 'buckets_count': 5, 'conflicts': []}, buckets)
# для эксперимента необходимо больше бакетов, чем доступно (5 > 4)
# success, buckets = False, [[], [], [], []]

success, buckets = add_experiment({'id': 1, 'buckets_count': 4, 'conflicts': [4]}, buckets)
# success, buckets = True, [[1], [1], [1], [1]]

success, buckets = add_experiment({'id': 2, 'buckets_count': 2, 'conflicts': [3]}, buckets)
# эксперимент с id=2 может быть в любых двух бакетах их четырёх
# success, buckets = True, [[1, 2], [1], [1, 2], [1]]

success, buckets = add_experiment({'id': 3, 'buckets_count': 2, 'conflicts': [2]}, buckets)
# можем добавить в бакеты, где не запущен экперимент с id=2
# success, buckets = True, [[1, 2], [1, 3], [1, 2], [1, 3]]

success, buckets = add_experiment({'id': 4, 'buckets_count': 1, 'conflicts': [1]}, buckets)
# не можем добавить, так как во всех бакетах запущен эксперимент, с которым конфликт
# success, buckets = False, [[1, 2], [1, 3], [1, 2], [1, 3]]

**Решение**

In [9]:
def add_experiment(experiment, buckets):
    """Проверяет можно ли добавить эксперимент, добавляет если можно.

    Распределять эксперименты нужно так, чтобы умещалось как можно больше экспериментов.

    :param experiment (dict): параметры эксперимента, который нужно запустить.
        Ключи словаря:
            - id - идентификатор эксперимента.
            - buckets_count - необходимое количество бакетов.
            - conflicts - список идентификаторов экспериментов, которые нельзя проводить
                одновременно на одних и тех же пользователях.
    :param buckets (list[list[int]]): список бакетов, в каждом бакете перечислены
            идентификаторы экспериментов, которые в нём проводятся.

    :return (success, buckets):
        success (boolean) - можно ли добавить эксперимент, True - можно, иначе - False
        buckets (list[list[int]]) - обновлённый список бакетов с добавленным экспериментом,
            если эксперимент добавить можно.
    """
    available_buckets = []
    for bucket_id, bucket in enumerate(buckets):
        if set(experiment['conflicts']) & set(bucket):  
            continue
        available_buckets.append((bucket_id, len(bucket)))
    
    if len(available_buckets) < experiment['buckets_count']:  
        return False, buckets
    
    sorted_available_buckets = sorted(available_buckets, key=lambda x: -x[1])
    
    for bucket_id, _ in sorted_available_buckets[:experiment['buckets_count']]:  
        buckets[bucket_id].append(experiment['id'])  
    
    return True, buckets

Проверка

In [10]:
total_buckets_count = 4
buckets = [[] for _ in range(total_buckets_count)]
success, buckets = add_experiment({'id': 0, 'buckets_count': 5, 'conflicts': []}, buckets)
# для эксперимента необходимо больше бакетов, чем доступно (5 > 4)
# success, buckets = False, [[], [], [], []]

### Task 2.

Реализуйте функцию process_user.

In [ ]:
import hashlib


def get_hash_modulo(value: str, modulo: int, salt: str):
    """Вычисляем остаток от деления: (hash(value + salt)) % modulo."""
    hash_value = int(hashlib.md5(str.encode(value + salt)).hexdigest(), 16)
    return hash_value % modulo

def process_user(user_id, buckets, experiments, bucket_salt):
    """Определяет в какие эксперименты попадает пользователь.

    Сначала нужно определить бакет пользователя.
    Затем для каждого эксперимента в этом бакете выбрать пилотную или контрольную группу.

    :param user_id (str): идентификатор пользователя
    :param buckets (list[list[int]]): список бакетов, в каждом бакете перечислены
            идентификаторы экспериментов, которые в нём проводятся.
    :param experiments (list[dict]): список словарей с информацией об экспериментах.
        Ключи словарей:
        - id (int) - идентификатор эксперимента
        - salt (str) - соль эксперимента для распределения пользователей на
            контрольную/пилотную группы.
    :param bucket_salt (str): соль для разбиения пользователей по бакетам.
        При одной соли каждый пользователь должен всегда попадать в один и тот же бакет.
        Если изменить соль, то распределение людей по бакетам должно измениться.
    :return bucket_id, experiment_groups:
        - bucket_id (int) - номер бакета (индекс элемента в buckets)
        - experiment_groups (list[tuple]) - список пар: id эксперимента, группа.
            Группы: 'A', 'B'.
        Пример: (8, [(194, 'A'), (73, 'B')])
    """
        # YOUR_CODE_HERE

**Пример**

In [ ]:
user_id = '1001'
experiments = [{'id': 0, 'salt': '0'}, {'id': 1, 'salt': '1'}]
buckets = [[0, 1], [1], []]
bucket_salt = 'a2N4'
bucket_id, experiment_groups = process_user(user_id, buckets, experiments, bucket_salt)
# В зависимости от значений bucket_salt и солей экспериментов, можно получить один из вариантов:
# bucket_id, experiment_groups = 0, [(0, 'A'), (1, 'A')]
# bucket_id, experiment_groups = 0, [(0, 'A'), (1, 'B')]
# bucket_id, experiment_groups = 0, [(0, 'B'), (1, 'A')]
# bucket_id, experiment_groups = 0, [(0, 'B'), (1, 'B')]
# bucket_id, experiment_groups = 1, [(1, 'A')]
# bucket_id, experiment_groups = 1, [(1, 'B')]
# bucket_id, experiment_groups = 2, []

**Решение**

In [25]:
import hashlib
from hashlib import md5


def get_hash_modulo(value: str, modulo: int, salt: str):
    """Вычисляем остаток от деления: (hash(value + salt)) % modulo."""
    hash_value = int(hashlib.md5(str.encode(value + salt)).hexdigest(), 16)
    return hash_value % modulo

def process_user(user_id, buckets, experiments, bucket_salt):
    """Определяет в какие эксперименты попадает пользователь.

    Сначала нужно определить бакет пользователя.
    Затем для каждого эксперимента в этом бакете выбрать пилотную или контрольную группу.

    :param user_id (str): идентификатор пользователя
    :param buckets (list[list[int]]): список бакетов, в каждом бакете перечислены
            идентификаторы экспериментов, которые в нём проводятся.
    :param experiments (list[dict]): список словарей с информацией об экспериментах.
        Ключи словарей:
        - id (int) - идентификатор эксперимента
        - salt (str) - соль эксперимента для распределения пользователей на
            контрольную/пилотную группы.
    :param bucket_salt (str): соль для разбиения пользователей по бакетам.
        При одной соли каждый пользователь должен всегда попадать в один и тот же бакет.
        Если изменить соль, то распределение людей по бакетам должно измениться.
    :return bucket_id, experiment_groups:
        - bucket_id (int) - номер бакета (индекс элемента в buckets)
        - experiment_groups (list[tuple]) - список пар: id эксперимента, группа.
            Группы: 'A', 'B'.
        Пример: (8, [(194, 'A'), (73, 'B')])
    """
    concat = user_id + bucket_salt
    bucket_id = int(hashlib.md5(concat.encode()).hexdigest(), 16) % len(buckets)
    experiment_groups = []
    for exp_id in buckets[bucket_id]:
        exp_dict = next((exp for exp in experiments if exp['id'] == exp_id), None)
        if exp_dict is None:
            continue  
        user_group = int(md5((user_id + exp_dict['salt']).encode()).hexdigest(), 16) % 2
        if user_group == 0:
            experiment_groups.append((exp_id, 'A'))
        else:
            experiment_groups.append((exp_id, 'B'))
    return bucket_id, experiment_groups

Проверка

In [27]:
user_id = '1001'
experiments = [{'id': 0, 'salt': '0'}, {'id': 1, 'salt': '1'}]
buckets = [[0, 1], [1], []]
bucket_salt = 'a2N4'
bucket_id, experiment_groups = process_user(user_id, buckets, experiments, bucket_salt)
print(f"bucket_id, experiment_groups = {bucket_id}, {experiment_groups}")

bucket_id, experiment_groups = 0, [(0, 'A'), (1, 'A')]
